In [2]:
# Nacitanie dat — WBD_suicide.csv
raw <- read.table("Data/WBD_suicide.csv", sep=";", header=TRUE,
                  check.names=FALSE, stringsAsFactors=FALSE,
                  na.strings=c("..", ""))

names(raw) <- c("Year", "YearCode", "Country", "CountryCode",
                "LifeExp", "GDP", "Alcohol", "Gini", "Suicide")

# Ponechame len uplne riadky
df <- raw[, c("Country", "LifeExp", "GDP", "Alcohol", "Gini", "Suicide")]
df <- na.omit(df)
rownames(df) <- NULL

cat("Rozmer:", nrow(df), "x", ncol(df), "\n")
str(df)

Rozmer: 51 x 6 
'data.frame':	51 obs. of  6 variables:
 $ Country: chr  "Albania" "Argentina" "Armenia" "Austria" ...
 $ LifeExp: num  79.5 76.8 76.2 81.9 74.2 ...
 $ GDP    : num  16442 23517 14976 60355 22302 ...
 $ Alcohol: num  5.11 8.04 4.98 11.97 10.9 ...
 $ Gini   : num  30.1 43.3 30 30.2 25.3 27.2 41.6 53.5 40.3 31.8 ...
 $ Suicide: num  3.78 9.28 2.7 14.58 17.52 ...
 - attr(*, "na.action")= 'omit' Named int [1:114] 1 3 4 5 6 7 10 11 13 14 ...
  ..- attr(*, "names")= chr [1:114] "1" "3" "4" "5" ...


In [3]:
# MNS model: Suicide ~ LifeExp + GDP + Alcohol + Gini

Y <- df$Suicide
X <- cbind(1, df$LifeExp, df$GDP, df$Alcohol, df$Gini)
colnames(X) <- c("intercept", "LifeExp", "GDP", "Alcohol", "Gini")

n <- nrow(X)
k <- ncol(X)

# odhad beta
betaHAT <- solve(t(X) %*% X) %*% t(X) %*% Y
rownames(betaHAT) <- colnames(X)
cat("Odhady beta:\n"); print(betaHAT)

# rezidua a odhad sigma^2
YHAT   <- X %*% betaHAT
epsHAT <- Y - YHAT
s2     <- as.numeric(t(epsHAT) %*% epsHAT / (n - k))
cat("\nOdhad sigma^2 (s2):", s2, "\n")
cat("Odhad sigma  (s) :", sqrt(s2), "\n")

# R^2, RSS, ESS, TSS
RSS <- sum(epsHAT^2)
TSS <- sum((Y - mean(Y))^2)
ESS <- TSS - RSS
R2  <- ESS / TSS
cat("\nRSS:", RSS, " ESS:", ESS, " TSS:", TSS, "\n")
cat("R^2:", R2, "\n")

# overenie cez lm()
model <- lm(Suicide ~ LifeExp + GDP + Alcohol + Gini, data = df)
summary(model)

Odhady beta:
                   [,1]
intercept  3.198117e+01
LifeExp   -2.530450e-01
GDP        5.483887e-05
Alcohol    4.721380e-01
Gini      -1.782976e-01

Odhad sigma^2 (s2): 37.77295 
Odhad sigma  (s) : 6.14597 

RSS: 1737.556  ESS: 374.5915  TSS: 2112.147 
R^2: 0.177351 


In [ ]:
# Test kontrastu: H0: beta_Alcohol = 2 * beta_Gini
# t.j. beta_Alcohol - 2*beta_Gini = 0
# poradie: (intercept, LifeExp, GDP, Alcohol, Gini)

a <- c(0, 0, 0, 1, -2)
r <- 0

X <- model.matrix(model)
T <- (t(a) %*% betaHAT - r) / sqrt(s2 * t(a) %*% solve(t(X)%*%X) %*% a)
krit <- qt(0.975, df = n-k)
p    <- 2 * (1 - pt(abs(T), df = n-k))

cat("T-statistika:    ", round(T, 4), "\n")
cat("Kritická hodnota:", round(krit, 4), "\n")
cat("p-hodnota:       ", round(p, 4), "\n\n")
if (p < 0.05) {
  cat("Záver: H0 ZAMIETAME (p < 0.05): beta_Alcohol != 2 * beta_Gini\n")
} else {
  cat("Záver: H0 NEZAMIETAME (p >= 0.05): nedostatok dôkazov proti beta_Alcohol = 2 * beta_Gini\n")
}

In [ ]:
# Test kontrastu: H0: beta_LifeExp = beta_GDP + beta_Alcohol
# t.j. beta_LifeExp - beta_GDP - beta_Alcohol = 0
# poradie: (intercept, LifeExp, GDP, Alcohol, Gini)

a <- c(0, 1, -1, -1, 0)
r <- 0

X <- model.matrix(model)
T <- (t(a) %*% betaHAT - r) / sqrt(s2 * t(a) %*% solve(t(X)%*%X) %*% a)
krit <- qt(0.975, df = n-k)
p    <- 2 * (1 - pt(abs(T), df = n-k))

cat("T-statistika:    ", round(T, 4), "\n")
cat("Kritická hodnota:", round(krit, 4), "\n")
cat("p-hodnota:       ", round(p, 4), "\n\n")
if (p < 0.05) {
  cat("Záver: H0 ZAMIETAME (p < 0.05): beta_LifeExp != beta_GDP + beta_Alcohol\n")
} else {
  cat("Záver: H0 NEZAMIETAME (p >= 0.05): nedostatok dôkazov proti beta_LifeExp = beta_GDP + beta_Alcohol\n")
}

In [ ]:
# Bonferroniho simultánne IS pre m=3 kontrasty
# poradie parametrov: (intercept, LifeExp, GDP, Alcohol, Gini)
#
# Kontrasty:
#   K1: beta_Alcohol - beta_GDP    → je alkohol dôležitejší ako HDP?
#   K2: beta_Alcohol - beta_Gini   → je alkohol dôležitejší ako nerovnosť?
#   K3: beta_GDP     - beta_Gini   → je HDP dôležitejší ako nerovnosť?

alpha <- 0.05
m     <- 3

A <- list(
  K1 = c(0,  0, -1,  1,  0),   # Alcohol - GDP
  K2 = c(0,  0,  0,  1, -1),   # Alcohol - Gini
  K3 = c(0,  0,  1,  0, -1)    # GDP     - Gini
)

X      <- model.matrix(model)
covMAT <- s2 * solve(t(X) %*% X)
krit_B <- qt(1 - alpha / (2 * m), df = n - k)

cat("Bonferroni kvantil (alpha=0.05, m=3, df=", n-k, "):", round(krit_B, 4), "\n")
cat("(oproti bežnému qt(0.975, df=", n-k, "):", round(qt(0.975, df=n-k), 4), ")\n\n")

for (nm in names(A)) {
  a     <- A[[nm]]
  est   <- as.numeric(t(a) %*% betaHAT)
  se    <- sqrt(as.numeric(t(a) %*% covMAT %*% a))
  dolna <- est - krit_B * se
  horna <- est + krit_B * se

  cat("---", nm, "---\n")
  cat("  Odhadnutý kontrast a'beta:", round(est,   4), "\n")
  cat("  Bonferroni IS 95%:       [", round(dolna, 4), ",", round(horna, 4), "]\n")
  if (dolna > 0) {
    cat("  Záver: IS neobsahuje 0, dolná hranica > 0: prvý faktor má štatisticky väčší vplyv\n\n")
  } else if (horna < 0) {
    cat("  Záver: IS neobsahuje 0, horná hranica < 0: druhý faktor má štatisticky väčší vplyv\n\n")
  } else {
    cat("  Záver: IS obsahuje 0: nedostatok dôkazov pre rozdiel medzi faktormi na hladine 5%\n\n")
  }
}

Test o heteroskedasticite

In [ ]:
# Whiteov test heteroskedasticity
# Model: Suicide ~ LifeExp + GDP + Alcohol + Gini
epsilonHAT <- resid(model)

artificial <- lm(epsilonHAT^2 ~
                   LifeExp + GDP + Alcohol + Gini +
                   I(LifeExp^2) + I(GDP^2) + I(Alcohol^2) + I(Gini^2) +
                   I(LifeExp * GDP) + I(LifeExp * Alcohol) + I(LifeExp * Gini) +
                   I(GDP * Alcohol) + I(GDP * Gini) +
                   I(Alcohol * Gini),
                 data = df)

n    <- nrow(df)
k    <- length(coef(model))   # pocet parametrov vrátane interceptu
cosi <- k * (k + 1) / 2       # pocet parametrov v artificial modeli

# Asymptoticky chi-square test
R2_art <- summary(artificial)$r.squared
W      <- n * R2_art
p_chi  <- 1 - pchisq(W, df = cosi - 1)

cat("=== WHITEOV TEST (chi-square) ===\n")
cat("Statistika W = n*R²:", round(W, 4), "\n")
cat("p-hodnota:          ", round(p_chi, 4), "\n")
cat("Záver:", ifelse(p_chi < 0.05,
  "ZAMIETAME H0: heteroskedasticita prítomná.",
  "NEZAMIETAME H0: heteroskedasticita sa nepreukázala."), "\n\n")

# Exaktny F-test
RSS    <- summary(artificial)$sigma^2 * (n - cosi)
RSSsub <- summary(lm(epsilonHAT^2 ~ 1))$sigma^2 * (n - 1)
Fprime <- ((RSSsub - RSS) / (cosi - 1)) / (RSS / (n - cosi))
p_F    <- 1 - pf(Fprime, df1 = cosi - 1, df2 = n - cosi)

cat("=== EXAKTNY F-TEST ===\n")
cat("F-statistika:", round(Fprime, 4), "\n")
cat("p-hodnota:   ", round(p_F, 4), "\n")
cat("Záver:", ifelse(p_F < 0.05,
  "ZAMIETAME H0: heteroskedasticita prítomná.",
  "NEZAMIETAME H0: heteroskedasticita sa nepreukázala."), "\n")

Test normality reziduí

In [ ]:
# Kolmogorov-Smirnov test normality reziduí
# H0: rezídua pochádzajú z normálneho rozdelenia
# H1: rezídua nepochádzajú z normálneho rozdelenia

epsilonHAT <- resid(model)

ks <- ks.test(epsilonHAT, "pnorm", mean = 0, sd = sqrt(var(epsilonHAT)))

cat("=== KS TEST NORMALITY ===\n")
cat("Statistika D:", round(ks$statistic, 4), "\n")
cat("p-hodnota:  ", round(ks$p.value, 4), "\n")
cat("Záver:", ifelse(ks$p.value < 0.05,
  "ZAMIETAME H0: rezídua nie sú normálne rozdelené.",
  "NEZAMIETAME H0: rezídua sú kompatibilné s normálnym rozdelením."), "\n")

# QQ plot
qqnorm(epsilonHAT, main = "QQ plot reziduí")
qqline(epsilonHAT, col = "red")

## Bonferroniho grafy

In [ ]:
# Graf 1: Forest plot Bonferroniho IS
alpha  <- 0.05
m      <- 3
A <- list(
  K1 = c(0,  0, -1,  1,  0),
  K2 = c(0,  0,  0,  1, -1),
  K3 = c(0,  0,  1,  0, -1)
)
X_mat  <- model.matrix(model)
covMAT <- s2 * solve(t(X_mat) %*% X_mat)
krit_B <- qt(1 - alpha / (2 * m), df = n - k)

ests <- sapply(A, function(a) as.numeric(t(a) %*% betaHAT))
ses  <- sapply(A, function(a) sqrt(as.numeric(t(a) %*% covMAT %*% a)))
lo_B <- ests - krit_B * ses
hi_B <- ests + krit_B * ses

kontrasty <- c("K1: Alcohol - GDP", "K2: Alcohol - Gini", "K3: GDP - Gini")
y_pos <- 3:1

par(mar = c(5, 13, 4, 2))
plot(ests, y_pos,
     xlim = range(c(lo_B, hi_B, 0)) + c(-0.1, 0.1),
     ylim = c(0.5, 3.5), yaxt = "n", pch = 19, col = "steelblue",
     xlab = "Odhadnuty kontrast a'beta", ylab = "",
     main = "Graf 1: Forest plot — Bonferroniho IS (m=3, alpha=5%)")
axis(2, at = y_pos, labels = kontrasty, las = 1)
segments(lo_B, y_pos, hi_B, y_pos, col = "steelblue", lwd = 2)
abline(v = 0, lty = 2, col = "red")

In [ ]:
# Graf 2: Stlpcovy graf kontrastov s Bonferroniho intervalmi chyb
alpha  <- 0.05
m      <- 3
A <- list(
  K1 = c(0,  0, -1,  1,  0),
  K2 = c(0,  0,  0,  1, -1),
  K3 = c(0,  0,  1,  0, -1)
)
X_mat  <- model.matrix(model)
covMAT <- s2 * solve(t(X_mat) %*% X_mat)
krit_B <- qt(1 - alpha / (2 * m), df = n - k)

ests <- sapply(A, function(a) as.numeric(t(a) %*% betaHAT))
ses  <- sapply(A, function(a) sqrt(as.numeric(t(a) %*% covMAT %*% a)))
lo_B <- ests - krit_B * ses
hi_B <- ests + krit_B * ses

kontrasty <- c("K1\nAlcohol-GDP", "K2\nAlcohol-Gini", "K3\nGDP-Gini")
farby <- ifelse(lo_B > 0 | hi_B < 0, "tomato", "steelblue")

bp <- barplot(ests, names.arg = kontrasty, col = farby,
              ylim = range(c(lo_B - 0.05, hi_B + 0.05, 0)),
              main = "Graf 2: Bonferroniho IS — stlpcovy graf",
              ylab = "Odhadnuty kontrast a'beta",
              border = "white")
arrows(bp, lo_B, bp, hi_B, angle = 90, code = 3, length = 0.1, lwd = 2)
abline(h = 0, lty = 2, col = "red")

In [ ]:
# Graf 3: Porovnanie Bonferroniho IS vs. standardneho 95% IS
alpha  <- 0.05
m      <- 3
A <- list(
  K1 = c(0,  0, -1,  1,  0),
  K2 = c(0,  0,  0,  1, -1),
  K3 = c(0,  0,  1,  0, -1)
)
X_mat  <- model.matrix(model)
covMAT <- s2 * solve(t(X_mat) %*% X_mat)
krit_B <- qt(1 - alpha / (2 * m), df = n - k)
krit_N <- qt(0.975, df = n - k)

ests <- sapply(A, function(a) as.numeric(t(a) %*% betaHAT))
ses  <- sapply(A, function(a) sqrt(as.numeric(t(a) %*% covMAT %*% a)))
lo_B <- ests - krit_B * ses;  hi_B <- ests + krit_B * ses
lo_N <- ests - krit_N * ses;  hi_N <- ests + krit_N * ses

kontrasty <- c("K1: Alcohol-GDP", "K2: Alcohol-Gini", "K3: GDP-Gini")
y_B <- c(3.15, 2.15, 1.15)
y_N <- c(2.85, 1.85, 0.85)

par(mar = c(5, 13, 4, 2))
plot(ests, c(3, 2, 1),
     xlim = range(c(lo_B, hi_B, lo_N, hi_N)) + c(-0.1, 0.1),
     ylim = c(0.5, 3.5), yaxt = "n", pch = 19,
     xlab = "Odhadnuty kontrast a'beta", ylab = "",
     main = "Graf 3: Bonferroni vs. standardny 95% IS")
axis(2, at = 3:1, labels = kontrasty, las = 1)
segments(lo_B, y_B, hi_B, y_B, col = "steelblue", lwd = 2)
segments(lo_N, y_N, hi_N, y_N, col = "tomato",    lwd = 2)
points(ests, c(3, 2, 1), pch = 19)
abline(v = 0, lty = 2, col = "gray40")

## Elipsa spolahlivosti pre (beta_Alcohol, beta_Gini) — styl letenky.R

In [ ]:
# Elipsoid spolahlivosti pre (beta_Alcohol, beta_Gini)
# poradie: (intercept)=1, LifeExp=2, GDP=3, Alcohol=4, Gini=5

alpha <- 0.05

X_mat   <- model.matrix(model)
covMAT  <- s2 * solve(t(X_mat) %*% X_mat)
covMAT2 <- covMAT[4:5, 4:5]
bHAT2   <- betaHAT[4:5, 1]

K95 <- 2 * qf(0.95, df1 = 2, df2 = n - k)
K99 <- 2 * qf(0.99, df1 = 2, df2 = n - k)

# Parametricka elipsa cez eigendekompoziciu covMAT2
eig   <- eigen(covMAT2)
theta <- seq(0, 2 * pi, length.out = 500)
circ  <- rbind(cos(theta), sin(theta))

ell95 <- bHAT2 + eig$vectors %*% (sqrt(K95 * eig$values) * circ)
ell99 <- bHAT2 + eig$vectors %*% (sqrt(K99 * eig$values) * circ)

xlim <- range(ell99[1, ])
ylim <- range(ell99[2, ])

# ---- Graf 1: 2 oblasti spolahlivosti ----
plot(ell99[1, ], ell99[2, ], type = "l", col = "grey", lwd = 2,
     xlab = "beta_Alcohol", ylab = "beta_Gini",
     xlim = xlim, ylim = ylim,
     main = "Oblast spolahlivosti pre (beta_Alcohol, beta_Gini)")
lines(ell95[1, ], ell95[2, ], col = "gray55", lwd = 2)
points(bHAT2[1], bHAT2[2], pch = 19, col = "red")
dev.copy(png, "graf_oblasti_spolahlivosti.png", width = 800, height = 600); dev.off()

# ---- Graf 2: elipsa + IS + Bonferroni ----
plot(ell95[1, ], ell95[2, ], type = "l", col = "grey", lwd = 2,
     xlab = "beta_Alcohol", ylab = "beta_Gini",
     xlim = xlim, ylim = ylim,
     main = "Elipsa + IS + Bonferroni pre (beta_Alcohol, beta_Gini)")
points(bHAT2[1], bHAT2[2], pch = 19, col = "red")

# Individualne IS
K0 <- qt(1 - alpha / 2, df = n - k) * sqrt(covMAT2[1, 1])
K1 <- qt(1 - alpha / 2, df = n - k) * sqrt(covMAT2[2, 2])
LA <- bHAT2[1] - K0;  UA <- bHAT2[1] + K0
LG <- bHAT2[2] - K1;  UG <- bHAT2[2] + K1

points(c(LA, UA), c(bHAT2[2], bHAT2[2]), type = "l", lwd = 2, col = "red")
points(c(bHAT2[1], bHAT2[1]), c(LG, UG),  type = "l", lwd = 2, col = "red")

# Obdlznik individualnych IS
points(c(LA, UA), c(LG, LG), type = "l", lty = "dashed", lwd = 2, col = "blue")
points(c(LA, UA), c(UG, UG), type = "l", lty = "dashed", lwd = 2, col = "blue")
points(c(LA, LA), c(LG, UG), type = "l", lty = "dashed", lwd = 2, col = "blue")
points(c(UA, UA), c(LG, UG), type = "l", lty = "dashed", lwd = 2, col = "blue")

# Bonferroniho obdlznik
B0 <- qt(1 - alpha / (2 * 2), df = n - k) * sqrt(covMAT2[1, 1])
B1 <- qt(1 - alpha / (2 * 2), df = n - k) * sqrt(covMAT2[2, 2])
LA_B <- bHAT2[1] - B0;  UA_B <- bHAT2[1] + B0
LG_B <- bHAT2[2] - B1;  UG_B <- bHAT2[2] + B1

points(c(LA_B, UA_B), c(LG_B, LG_B), type = "l", lwd = 2, col = "green")
points(c(LA_B, UA_B), c(UG_B, UG_B), type = "l", lwd = 2, col = "green")
points(c(LA_B, LA_B), c(LG_B, UG_B), type = "l", lwd = 2, col = "green")
points(c(UA_B, UA_B), c(LG_B, UG_B), type = "l", lwd = 2, col = "green")
dev.copy(png, "graf_elipsa_bonferroni.png", width = 800, height = 600); dev.off()

cat("Grafy ulozene:\n  graf_oblasti_spolahlivosti.png\n  graf_elipsa_bonferroni.png\n")